# CS 3892 / 5892 — Session 19 · Lean 4 I: terms, types, tactics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-11-03-lean-4-terms-types-tactics.ipynb)

**Tuesday, November 3, 2026.** Lean 4 is a programming language whose type checker also checks
proofs. A proposition is a type and a proof is a program of that type (**Curry–Howard**). **Tactics**
build that program one step at a time, and Lean shows the goal after every step. **`sorry`** compiles
and proves nothing; **`#print axioms`** says what a theorem rests on.

Every Lean snippet on the slides is a region of a file in `lean/S19/`, and every output the slides
show is asserted by a script in `python/`. Lean v4.33.0, core library only: the version HW4 uses.
Colab has no Lean editor, so you will not see goals as you type; the scripts print them. For that
view, open `lean/` in VS Code with the Lean 4 extension.

## Setup

Run these two once. The second installs Lean: give it a minute or two.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-11-03-lean-4-terms-types-tactics"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
LD  = ROOT / "sessions" / SESSION / "lean"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- Lean v4.33.0, installed by HW4's checker: elan (the Lean version manager, pinned by
# SHA-256) into ~/.elan, then Lean through it -- about 575 MB, a minute or two the first time,
# and nothing to install after that. A Lean you already have is used as it is.
out = run(PYD / "lean_cli.py")
assert "version 4.33.0" in out

## 1. Lean is a programming language: the running counter

`step` is one move of the running counter (below 10, count up; otherwise reset to 0), and `run 0 n`
is where it is after `n` moves. `#eval` runs them. Then three theorems: `step_3` by computation,
`step_le` for every `x` (Thursday's "preserved" condition), and `run_le` by induction, for every
number of moves. The script prints Lean's goal before every tactic line.

In [ ]:
display(SVG(filename=str(FIG / "counter.svg")))
out = run(PYD / "01_lean_is_a_language.py")
assert "[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 0, 1]" in out and "proved" in out

## 2. Curry–Howard: propositions are types, proofs are programs

`and_swap` proves `A ∧ B → B ∧ A` with a function on pairs; `swap` is the same function on data.
`and_swap'` proves it again with tactics, and the script prints the goal at each step.

In [ ]:
out = run(PYD / "02_curry_howard.py")
assert "S19.and_swap (A B : Prop) : A ∧ B → B ∧ A" in out and "prove the same proposition" in out

## 3. Tactics that close a goal, and one that gives up

`rfl`, `decide`, `omega` and `simp`; then `omega` on `x ≤ x * x`, which is true but not linear.

In [ ]:
out = run(PYD / "03_tactics.py")
assert "omega could not prove the goal" in out

## 4. `sorry` compiles, and proves nothing

A `sorry`, a home-made `axiom`, and an honest proof, each read with `#print axioms`.

In [ ]:
out = run(PYD / "04_sorry_and_axioms.py")
assert "sorryAx" in out and "[all_zero]" in out and "does not depend on any axioms" in out

## 5. Sets are predicates: a constructive proof, and a classical one

De Morgan for union, `(A ∪ B)ᶜ = Aᶜ ∩ Bᶜ`, never asks whether `x ∈ A` holds. `¬¬p → p` has to
(`by_cases`), and `#print axioms` shows `Classical.choice` appear.

In [ ]:
out = run(PYD / "05_sets_as_predicates.py")
assert "Classical.choice" in out and "[propext, Quot.sound]" in out

## 6. The whole project builds

`lake build`, as VS Code runs it: every file in the library, no `sorry`, no warning.

In [ ]:
show(LD / "S19.lean")
out = run(PYD / "06_whole_library.py")
assert "Build completed successfully" in out

## Try it yourself

Open the folder icon (left) → `cs3892-examples/sessions/cs3892-2026-11-03-lean-4-terms-types-tactics/lean/S19/`,
double-click a file, edit, save (Ctrl+S), and re-run its cell. A runtime is deleted when it
disconnects: download anything you want to keep.

- In `Counter.lean`, make the counter count to 12 (`x < 12`). Which theorem breaks, at which line,
  and what goal does Lean say it cannot close?
- In `CurryHoward.lean`, prove `A ∨ B → B ∨ A`: once as a term (`Or.inl`, `Or.inr`, `match`), once
  with tactics (`intro`, `cases h with`, `left`, `right`).
- In `Sorry.lean`, replace the `sorry` in `bogus` with `decide`. What does Lean say now?
- In `Sets.lean`, try to prove `not_not` without `by_cases`. Where do you get stuck?